# DSPy SQL Agent — Versão Aprimorada
> **Equipe:** Cauã (Signatures) · Davi (Modules) · João Paulo (Otimizadores) · Kauan (Features extras) - 3°ADS

Este notebook implementa um agente de consultas SQL em linguagem natural usando **DSPy**,
com signatures avançadas, módulos robustos com *MultiChainComparison*, otimizadores
`BootstrapFewShot` + `MIPROv2`, cache de queries, histórico de execução e pipeline de avaliação completo.


## 1. Setup do Ambiente

In [1]:
# Instala dependências do sistema e Ollama
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Instala DSPy e dependências Python
!pip install -q dspy-ai colorama tabulate


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.3 MB/s eta 0:00:00

In [2]:
#Teste do servidor
# !ollama serve &
# !sleep 3
# !ollama list

In [3]:
import threading
import subprocess
import time

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL)

thread = threading.Thread(target=run_ollama_serve, daemon=True)
thread.start()
time.sleep(2)  # aguarda Ollama iniciar
print("✓ Ollama service iniciado em background")


✓ Ollama service iniciado em background


In [4]:
# Baixa o modelo
!ollama pull qwen2.5-coder:3b


## 2. Imports e Configuração Global

In [5]:
import dspy
import sqlite3
import requests
import json
import hashlib
import time
import logging
from datetime import datetime
from typing import Optional
from tabulate import tabulate

# Logging legível
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("dspy_agent")


In [6]:
def setup_ollama(model: str = "qwen2.5-coder:3b", temperature: float = 0.1) -> bool:
    """Verifica se o Ollama está rodando e configura o LM no DSPy."""
    try:
        resp = requests.get("http://localhost:11434/api/tags", timeout=5)
        resp.raise_for_status()
        logger.info("Ollama está rodando ✓")
    except Exception as e:
        logger.error(f"Ollama não acessível: {e}")
        return False

    lm = dspy.LM(
        f"ollama_chat/{model}",
        api_base="http://localhost:11434",
        temperature=temperature,
        max_tokens=512,
        cache=False,          # DSPy cache interno desativado; usamos cache próprio
    )
    dspy.configure(lm=lm)
    logger.info(f"DSPy configurado com modelo: {model}")
    return True

setup_ollama()


True

## 3. Banco de Dados de Exemplo

In [7]:
def create_sample_database(db_path: str = "company.db") -> None:
    """Cria e popula o banco de dados de demonstração."""
    conn = sqlite3.connect(db_path)
    c = conn.cursor()

    c.executescript("""
        CREATE TABLE IF NOT EXISTS employees (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            department TEXT NOT NULL,
            salary INTEGER NOT NULL,
            hire_date TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS departments (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            budget INTEGER NOT NULL,
            manager TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS projects (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            department_id INTEGER NOT NULL,
            budget INTEGER NOT NULL,
            status TEXT NOT NULL,
            FOREIGN KEY (department_id) REFERENCES departments(id)
        );
    """)

    employees = [
        (1, "Alice Johnson",  "Engineering", 95000, "2022-01-15"),
        (2, "Bob Smith",      "Sales",        75000, "2021-06-20"),
        (3, "Carol White",    "Engineering",  90000, "2023-02-10"),
        (4, "David Brown",    "Marketing",    70000, "2022-09-05"),
        (5, "Eve Davis",      "Engineering",  98000, "2020-03-12"),
        (6, "Frank Miller",   "Sales",        80000, "2021-11-01"),
        (7, "Grace Lee",      "Marketing",    72000, "2023-04-18"),
        (8, "Henry Adams",    "Engineering",  87000, "2022-07-22"),
        (9, "Irene Torres",   "HR",           65000, "2021-03-09"),
        (10,"James Wilson",   "HR",           63000, "2023-08-14"),
    ]
    departments = [
        (1, "Engineering", 500000, "Alice Johnson"),
        (2, "Sales",       300000, "Bob Smith"),
        (3, "Marketing",   200000, "Carol White"),
        (4, "HR",          150000, "Irene Torres"),
    ]
    projects = [
        (1, "API Development",   1, 150000, "In Progress"),
        (2, "Mobile App",        1, 200000, "Planning"),
        (3, "Sales Dashboard",   2,  80000, "Completed"),
        (4, "Brand Refresh",     3,  60000, "In Progress"),
        (5, "HR Portal",         4,  40000, "Planning"),
        (6, "Data Warehouse",    1, 180000, "In Progress"),
    ]

    c.executemany("INSERT OR IGNORE INTO employees VALUES (?,?,?,?,?)", employees)
    c.executemany("INSERT OR IGNORE INTO departments VALUES (?,?,?,?)", departments)
    c.executemany("INSERT OR IGNORE INTO projects VALUES (?,?,?,?,?)", projects)

    conn.commit()
    conn.close()
    logger.info("Banco de dados criado/atualizado ✓")

create_sample_database()


In [26]:
# Schema string reusado por várias signatures
DB_SCHEMA = """
Tables and columns:
  employees  : id (PK), name TEXT, department TEXT (FK linking to departments.name), salary INTEGER, hire_date TEXT
  departments: id (PK), name TEXT, budget INTEGER, manager TEXT
  projects   : id (PK), name TEXT, department_id INTEGER (FK linking to departments.id),
               budget INTEGER, status TEXT ('In Progress'|'Planning'|'Completed')
"""

def get_live_schema(db_path: str = "company.db") -> str:
    """Retorna schema real lido do SQLite (útil para bancos dinâmicos)."""
    conn = sqlite3.connect(db_path)
    rows = conn.execute(
        "SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()
    conn.close()
    return "\n".join(f"-- {name}\n{sql};" for name, sql in rows)

print(get_live_schema())


-- departments
CREATE TABLE departments (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            budget INTEGER NOT NULL,
            manager TEXT NOT NULL
        );
-- employees
CREATE TABLE employees (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            department TEXT NOT NULL,
            salary INTEGER NOT NULL,
            hire_date TEXT NOT NULL
        );
-- projects
CREATE TABLE projects (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            department_id INTEGER NOT NULL,
            budget INTEGER NOT NULL,
            status TEXT NOT NULL,
            FOREIGN KEY (department_id) REFERENCES departments(id)
        );


## 4. Signatures

Signatures definem o **contrato de entrada/saída** de cada etapa do pipeline.
Boas signatures têm:
- Docstring clara descrevendo a tarefa
- `desc=` preciso em cada campo
- Tipagem explícita com `str`, `int`, `list`, etc.

### Por que `MultiChainComparison` funciona aqui?
A pergunta original questionava se *multi-chain* seria viável.  
**Sim** — o módulo `dspy.MultiChainComparison` gera N cadeias de raciocínio independentes
e escolhe a melhor resposta por votação/comparação interna, reduzindo erros de SQL por
variação estocástica do modelo. É especialmente útil quando o LLM tende a alucinações em queries complexas.


In [27]:
# Signatures Aprimoradas

class GenerateSQL(dspy.Signature):
    """Translate a natural language question into a valid SQLite query.

    Rules:
    - Use only tables and columns listed in the schema.
    - Return ONLY the SQL statement, no explanation, no markdown fences.
    - Use LIMIT 100 by default unless the question asks for all rows.
    - Use standard SQLite syntax (no CTEs with RECURSIVE unless needed).
    """
    schema:   str = dspy.InputField(desc="Full database schema with table and column definitions")
    question: str = dspy.InputField(desc="Natural language question about the database")
    sql_query: str = dspy.OutputField(desc="Valid SQLite SELECT statement, nothing else")


class RefineSQL(dspy.Signature):
    """Fix a broken SQL query given the original question, the faulty query, and the error message.

    Rules:
    - Keep the same semantic intent as the original question.
    - Fix only what caused the error.
    - Return ONLY the corrected SQL statement.
    """
    schema:    str = dspy.InputField(desc="Database schema")
    question:  str = dspy.InputField(desc="Original natural language question")
    sql_query: str = dspy.InputField(desc="SQL query that produced an error")
    error:     str = dspy.InputField(desc="SQLite error message")
    refined_sql: str = dspy.OutputField(desc="Corrected SQL statement only")


class InterpretResults(dspy.Signature):
    """Interpret SQL query results and produce a concise, friendly natural-language answer.

    - If results are empty, say so clearly.
    - Round floats to 2 decimal places.
    - Be concise — at most 3 sentences.
    """
    question: str  = dspy.InputField(desc="The original question the user asked")
    sql_query: str = dspy.InputField(desc="The SQL query that was executed")
    results:   str = dspy.InputField(desc="Query results as a JSON-encoded list of rows")
    answer:    str = dspy.OutputField(desc="Friendly natural-language answer to the question")


class ValidateSQL(dspy.Signature):
    """Decide if a SQL query is safe and semantically correct before execution.

    Respond with is_valid=True only if:
    - The query uses ONLY SELECT.
    - Read-only SQLite functions like strftime, COUNT, SUM, AVG are perfectly VALID and SAFE.
    - NO destructive commands (INSERT, UPDATE, DELETE, DROP).
    """
    schema:    str  = dspy.InputField(desc="Database schema")
    sql_query: str  = dspy.InputField(desc="SQL query to evaluate")
    is_valid:  bool = dspy.OutputField(desc="True if safe to execute, False otherwise")
    reason:    str  = dspy.OutputField(desc="Short explanation")


class ClassifyQuestion(dspy.Signature):
    """Classify a database question by complexity.

    Categories:
    - 'simple'  : single-table lookup, no aggregation
    - 'medium'  : aggregation (COUNT/SUM/AVG) or single JOIN
    - 'complex' : multiple JOINs, subqueries, or window functions
    """
    question:   str = dspy.InputField(desc="Natural language question")
    complexity: str = dspy.OutputField(desc="One of: simple | medium | complex")
    reasoning:  str = dspy.OutputField(desc="One sentence explaining the classification")


print("✓ Signatures carregadas")


✓ Signatures carregadas


## 5. Modules

In [37]:
# Modules Aprimorados

class SQLValidator(dspy.Module):
    """Validador determinístico em Python puro. Não usa o LLM."""
    def __init__(self):
        super().__init__()
        # Palavras estritamente proibidas em consultas de leitura
        self.forbidden_keywords = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "REPLACE", "TRUNCATE"]

    def forward(self, schema: str, sql_query: str) -> dspy.Prediction:
        upper_sql = sql_query.upper()

        # Checa se alguma palavra proibida está na string do SQL
        for word in self.forbidden_keywords:
            if f" {word} " in upper_sql or upper_sql.startswith(word):
                return dspy.Prediction(
                    is_valid=False,
                    reason=f"Segurança: A query contém o comando destrutivo '{word}'."
                )

        return dspy.Prediction(is_valid=True, reason="Query de leitura segura.")

class SQLGeneratorSimple(dspy.Module):
    """Gerador simples com ChainOfThought — apenas gera o SQL."""
    def __init__(self):
        super().__init__()
        self.generator = dspy.ChainOfThought(GenerateSQL)

    def forward(self, question: str, schema: str) -> dspy.Prediction:
        pred = self.generator(schema=schema, question=question)
        sql = pred.sql_query.strip().strip('`') # Correção sintática
        return dspy.Prediction(sql_query=sql, strategy="simple")

class SQLGeneratorMultiChain(dspy.Module):
    """Gera M candidatos via ChainOfThought e usa o comparador para escolher o melhor."""
    def __init__(self, M: int = 3):
        super().__init__()
        self.M = M
        # 1. Módulo para GERAR as múltiplas respostas
        self.generator = dspy.ChainOfThought(GenerateSQL)
        # 2. Módulo para COMPARAR as respostas geradas
        self.comparator = dspy.MultiChainComparison(GenerateSQL)

    def forward(self, question: str, schema: str) -> dspy.Prediction:
        # 1. Gera M respostas independentes pedindo config(n=M) para o LLM
        raw_preds = self.generator(question=question, schema=schema, config=dict(n=self.M))

        # 2. Avalia e escolhe a melhor resposta passando as 'completions' obrigatoriamente
        best_pred = self.comparator(completions=raw_preds.completions, question=question, schema=schema)

        sql = best_pred.sql_query.strip().strip('`')
        return dspy.Prediction(sql_query=sql, strategy=f"multi_chain(M={self.M})")

class SQLRefiner(dspy.Module):
    """Refina uma query que falhou com base no erro do banco de dados."""
    def __init__(self):
        super().__init__()
        self.refiner = dspy.ChainOfThought(RefineSQL)

    def forward(self, question: str, schema: str, sql_query: str, error: str) -> dspy.Prediction:
        pred = self.refiner(schema=schema, question=question, sql_query=sql_query, error=error)
        sql = pred.refined_sql.strip().strip('`')
        return dspy.Prediction(sql_query=sql)

class SQLInterpreter(dspy.Module):
    """Converte resultados brutos em linguagem natural."""
    def __init__(self):
        super().__init__()
        self.interpret = dspy.ChainOfThought(InterpretResults)

    def forward(self, question: str, sql_query: str, results) -> str:
        results_json = json.dumps(results[:50])
        pred = self.interpret(question=question, sql_query=sql_query, results=results_json)
        return pred.answer

print("✓ Modules corrigidos e carregados")


✓ Modules corrigidos e carregados


## 6. Pipeline Principal do Agente

In [11]:
# Pipeline completo com segurança rigorosa

class QueryCache:
    def __init__(self):
        self._store: dict = {}
    def key(self, question: str) -> str:
        return hashlib.md5(question.strip().lower().encode()).hexdigest()
    def get(self, question: str):
        return self._store.get(self.key(question))
    def set(self, question: str, value):
        self._store[self.key(question)] = value
    def clear(self):
        self._store.clear()

class SQLAgent(dspy.Module):
    """
    Agente principal com execução segura.
    Fluxo: Classificar -> Gerar -> Validar -> (Se válido) Executar -> (Se erro) Refinar -> Validar -> Executar
    """
    def __init__(self, db_path: str = "company.db", use_cache: bool = True):
        super().__init__()
        self.db_path    = db_path
        self.schema     = DB_SCHEMA
        self.use_cache  = use_cache

        # Módulos instanciados
        self.classifier    = dspy.Predict(ClassifyQuestion)
        self.validator     = SQLValidator()
        self.gen_simple    = SQLGeneratorSimple()
        self.gen_multi     = SQLGeneratorMultiChain(M=3)
        self.refiner       = SQLRefiner()
        self.interpreter   = SQLInterpreter()

        self.cache   = QueryCache()
        self.history: list = []

    def _get_conn(self):
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        return conn

    def forward(self, question: str, interpret: bool = True) -> dict:
        start_time = time.time()

        if self.use_cache:
            cached = self.cache.get(question)
            if cached:
                logger.info("Cache hit para: %s", question[:60])
                return {**cached, "from_cache": True}

        entry = {
            "question":    question,
            "timestamp":   datetime.now().isoformat(),
            "from_cache":  False,
            "error":       None
        }

        conn = self._get_conn()
        try:
            # Classificação
            cls = self.classifier(question=question)
            complexity = cls.complexity.strip().lower()
            entry["complexity"] = complexity

            # Geração Pura (Roteamento único)
            # Devido à limitação do Ollama local não suportar o parâmetro 'n',
            # roteamos tudo para o gerador simples e confiamos no pipeline de Refinamento.
            gen_pred = self.gen_simple(question=question, schema=self.schema)

            sql_candidate = gen_pred.sql_query
            strategy = gen_pred.strategy

            # Função auxiliar de tentativa de execução
            def try_execute(sql_to_run):
                val = self.validator(schema=self.schema, sql_query=sql_to_run)
                entry["validation"] = {"is_valid": val.is_valid, "reason": val.reason}

                if not val.is_valid:
                    raise Exception(f"Bloqueado pelo validador: {val.reason}")

                # Execução
                return conn.execute(sql_to_run).fetchall()

            # Primeira tentativa
            try:
                raw_results = try_execute(sql_candidate)
                entry["sql_query"] = sql_candidate
                entry["strategy"]  = strategy
            except Exception as e:
                # Pipeline de Refinamento
                error_msg = str(e)
                logger.info(f"Tentativa inicial falhou, iniciando refinamento. Erro: {error_msg}")
                refine_pred = self.refiner(question=question, schema=self.schema, sql_query=sql_candidate, error=error_msg)
                sql_refined = refine_pred.sql_query

                try:
                    raw_results = try_execute(sql_refined)
                    entry["sql_query"] = sql_refined
                    entry["strategy"]  = strategy + "+refined"
                except Exception as e_refined:
                    entry["sql_query"] = sql_refined
                    entry["strategy"]  = strategy + "+refined_failed"
                    raise Exception(f"Falha mesmo após refinamento: {e_refined}")

            # Serialização e Interpretação
            results = [tuple(r) for r in raw_results]
            entry["results"] = results
            entry["result_count"] = len(results)

            if interpret:
                entry["answer"] = self.interpreter(
                    question=question,
                    sql_query=entry["sql_query"],
                    results=results,
                )
            else:
                entry["answer"] = None

        except Exception as final_error:
            entry["error"]  = str(final_error)
            entry["answer"] = f"Não foi possível resolver a consulta: {final_error}"
            logger.error("Erro no fluxo: %s", final_error)
        finally:
            conn.close()

        entry["elapsed_ms"] = round((time.time() - start_time) * 1000)

        self.history.append(entry)
        if self.use_cache and not entry.get("error"):
            self.cache.set(question, entry)

        return entry

agent = SQLAgent(use_cache=True)
print("✓ SQLAgent inicializado com pipeline seguro")

✓ SQLAgent inicializado com pipeline seguro


/usr/local/lib/python3.12/dist-packages/dspy/signatures/signature.py:185: UserWarning: Field name "schema" in "StringSignature" shadows an attribute in parent "Signature"
  cls = super().__new__(mcs, signature_name, bases, namespace, **kwargs)


## 7. Exemplos de Treinamento (Few-Shot)

In [12]:
def create_training_examples() -> list:
    """
    Exemplos anotados para BootstrapFewShot.
    Cada Example precisa de .with_inputs() para marcar quais campos são entradas.
    """
    examples = [
        # Simple
        dspy.Example(
            schema=DB_SCHEMA,
            question="How many employees are in the Engineering department?",
            sql_query="SELECT COUNT(*) AS total FROM employees WHERE department = 'Engineering'",
        ).with_inputs("schema", "question"),

        dspy.Example(
            schema=DB_SCHEMA,
            question="What is the highest salary in the company?",
            sql_query="SELECT MAX(salary) AS max_salary FROM employees",
        ).with_inputs("schema", "question"),

        dspy.Example(
            schema=DB_SCHEMA,
            question="List all employees with salary above 90000",
            sql_query="SELECT name, salary FROM employees WHERE salary > 90000 ORDER BY salary DESC",
        ).with_inputs("schema", "question"),

        # Medium
        dspy.Example(
            schema=DB_SCHEMA,
            question="Which department has the highest budget?",
            sql_query="SELECT name, budget FROM departments ORDER BY budget DESC LIMIT 1",
        ).with_inputs("schema", "question"),

        dspy.Example(
            schema=DB_SCHEMA,
            question="Show all projects in progress with their budget",
            sql_query="SELECT name, budget FROM projects WHERE status = 'In Progress' ORDER BY budget DESC",
        ).with_inputs("schema", "question"),

        dspy.Example(
            schema=DB_SCHEMA,
            question="What is the average salary per department?",
            sql_query="SELECT department, ROUND(AVG(salary), 2) AS avg_salary FROM employees GROUP BY department ORDER BY avg_salary DESC",
        ).with_inputs("schema", "question"),

        # Complex
        dspy.Example(
            schema=DB_SCHEMA,
            question="Which departments have more than 2 employees and what is their average salary?",
            sql_query="SELECT department, COUNT(*) AS headcount, ROUND(AVG(salary), 2) AS avg_salary FROM employees GROUP BY department HAVING COUNT(*) > 2",
        ).with_inputs("schema", "question"),

        dspy.Example(
            schema=DB_SCHEMA,
            question="List employees hired in 2022 along with their department budget",
            sql_query="SELECT e.name, e.hire_date, d.budget FROM employees e JOIN departments d ON e.department = d.name WHERE e.hire_date LIKE '2022%' ORDER BY e.hire_date",
        ).with_inputs("schema", "question"),

        dspy.Example(
            schema=DB_SCHEMA,
            question="Show total project budget per department",
            sql_query="SELECT d.name AS department, SUM(p.budget) AS total_project_budget FROM projects p JOIN departments d ON p.department_id = d.id GROUP BY d.name ORDER BY total_project_budget DESC",
        ).with_inputs("schema", "question"),

        dspy.Example(
            schema=DB_SCHEMA,
            question="Who is the highest paid employee in each department?",
            sql_query="SELECT department, name, salary FROM employees WHERE (department, salary) IN (SELECT department, MAX(salary) FROM employees GROUP BY department) ORDER BY salary DESC",
        ).with_inputs("schema", "question"),
    ]
    return examples


TRAIN_EXAMPLES = create_training_examples()
print(f"✓ {len(TRAIN_EXAMPLES)} exemplos de treinamento criados")


✓ 10 exemplos de treinamento criados


## 8. Métrica de Avaliação

In [13]:
def sql_accuracy_metric(example: dspy.Example, pred: dspy.Prediction, trace=None) -> float:
    """
    Métrica composta para avaliar qualidade do SQL gerado.

    Pontuação:
    - +0.5 se a query executa sem erro
    - +0.3 se o número de colunas retornadas bate com o esperado
    - +0.2 se as palavras-chave essenciais do SQL esperado estão presentes
    """
    score = 0.0
    predicted_sql = getattr(pred, "sql_query", "").strip()
    expected_sql  = example.sql_query.strip()

    if not predicted_sql:
        return 0.0

    # Tenta executar
    try:
        conn = sqlite3.connect("company.db")
        rows = conn.execute(predicted_sql).fetchall()
        conn.close()
        score += 0.5

        # Verifica número de colunas (compara com expected se disponível)
        try:
            conn2 = sqlite3.connect("company.db")
            exp_rows = conn2.execute(expected_sql).fetchall()
            conn2.close()
            if rows and exp_rows:
                if len(rows[0]) == len(exp_rows[0]):
                    score += 0.3
            elif not rows and not exp_rows:
                score += 0.3  # ambos vazios = correto
        except Exception:
            pass

    except Exception:
        pass

    # Verifica keywords essenciais (FROM, WHERE, GROUP BY, etc.)
    pred_upper = predicted_sql.upper()
    exp_upper  = expected_sql.upper()
    keywords = ["FROM", "WHERE", "GROUP BY", "ORDER BY", "JOIN", "HAVING",
                "COUNT", "SUM", "AVG", "MAX", "MIN"]
    exp_kw  = [kw for kw in keywords if kw in exp_upper]
    pred_kw = [kw for kw in exp_kw if kw in pred_upper]

    if exp_kw:
        score += 0.2 * (len(pred_kw) / len(exp_kw))
    else:
        score += 0.2  # sem keywords obrigatórias = bônus

    return round(score, 3)


# Teste rápido da métrica
class _FakePred:
    sql_query = "SELECT COUNT(*) AS total FROM employees WHERE department = 'Engineering'"

score = sql_accuracy_metric(TRAIN_EXAMPLES[0], _FakePred())
print(f"Teste da métrica (query correta): {score:.3f}  (esperado ≈ 1.0)")


Teste da métrica (query correta): 1.000  (esperado ≈ 1.0)


## 9. Otimizadores

### Estratégia de otimização
| Otimizador | Quando usar |
|---|---|
| `BootstrapFewShot` | Rápido, sem custo extra de LLM, injeta exemplos validados nos prompts |
| `BootstrapFewShotWithRandomSearch` | Quando há mais tempo e se quer maximizar a métrica com busca aleatória |
| `MIPROv2` | Otimiza *instruções* além dos exemplos — ideal para produção, mais caro |

> **OBS:** Otimizadores do DSPy requerem que o LM seja chamado várias vezes.
> Para Ollama local com modelo pequeno, `BootstrapFewShot` é o mais prático.


In [14]:
from dspy.teleprompt import BootstrapFewShot, BootstrapFewShotWithRandomSearch

# Otimizador 1: BootstrapFewShot (mais rápido)

def optimize_with_bootstrap(
    module_to_optimize: dspy.Module,
    trainset: list,
    metric,
    max_bootstrapped_demos: int = 4,
    max_labeled_demos: int = 4,
) -> dspy.Module:
    """
    Usa BootstrapFewShot para selecionar os melhores exemplos de few-shot
    automaticamente a partir do trainset.
    """
    logger.info("Iniciando BootstrapFewShot...")
    teleprompter = BootstrapFewShot(
        metric=metric,
        max_bootstrapped_demos=max_bootstrapped_demos,
        max_labeled_demos=max_labeled_demos,
    )
    optimized = teleprompter.compile(
        module_to_optimize,
        trainset=trainset,
    )
    logger.info("BootstrapFewShot concluído ✓")
    return optimized


def optimize_with_random_search(
    module_to_optimize: dspy.Module,
    trainset: list,
    metric,
    num_candidate_programs: int = 8,
    max_bootstrapped_demos: int = 4,
) -> dspy.Module:
    """
    Variante com busca aleatória — testa N configurações e retorna a melhor.
    Mais pesado, mas geralmente produz resultados superiores.
    """
    logger.info("Iniciando BootstrapFewShotWithRandomSearch (num_candidates=%d)...",
                num_candidate_programs)
    teleprompter = BootstrapFewShotWithRandomSearch(
        metric=metric,
        max_bootstrapped_demos=max_bootstrapped_demos,
        num_candidate_programs=num_candidate_programs,
    )
    optimized = teleprompter.compile(
        module_to_optimize,
        trainset=trainset,
    )
    logger.info("RandomSearch concluído ✓")
    return optimized


print("✓ Funções de otimização definidas")


✓ Funções de otimização definidas


In [15]:
# Otimizador 2: MIPROv2 (otimiza instruções + exemplos)

try:
    from dspy.teleprompt import MIPROv2

    def optimize_with_mipro(
        module_to_optimize: dspy.Module,
        trainset: list,
        metric,
        num_trials: int = 10,
        minibatch_size: int = 5,
    ) -> dspy.Module:
        """
        MIPROv2 otimiza tanto as *instruções* das Signatures quanto os exemplos.
        É o otimizador mais poderoso do DSPy para produção.

        Parâmetros recomendados para Ollama local:
        - num_trials=10 (suficiente para modelos menores)
        - minibatch_size=5 (evita sobrecarregar o Ollama)
        """
        logger.info("Iniciando MIPROv2 (trials=%d)...", num_trials)
        teleprompter = MIPROv2(
            metric=metric,
            num_trials=num_trials,
            minibatch_size=minibatch_size,
            verbose=True,
        )
        optimized = teleprompter.compile(
            module_to_optimize,
            trainset=trainset,
            requires_permission_to_run=False,
        )
        logger.info("MIPROv2 concluído ✓")
        return optimized

    print("✓ MIPROv2 disponível")

except ImportError:
    print("⚠ MIPROv2 não disponível nesta versão do dspy-ai — use BootstrapFewShot")
    def optimize_with_mipro(*args, **kwargs):
        raise NotImplementedError("MIPROv2 não disponível. Atualize: pip install -U dspy-ai")


✓ MIPROv2 disponível


## 10. Executando a Otimização

In [38]:
# Módulo base para otimizar
# Usamos SQLGeneratorSimple pois é o mais rápido de otimizar localmente
base_module = SQLGeneratorSimple()

# # Divide trainset em treino/validação (80/20)
split = int(len(TRAIN_EXAMPLES) * 0.8)
train_set = TRAIN_EXAMPLES[:split]
val_set   = TRAIN_EXAMPLES[split:]

print(f"Train: {len(train_set)} exemplos | Val: {len(val_set)} exemplos")

# Esta célula pode demorar vários minutos com Ollama local.
# Se quiser pular, rodar apenas: optimized_module = base_module (como ja está agora)

optimized_module = optimize_with_bootstrap(
    module_to_optimize=base_module,
    trainset=train_set,
    metric=sql_accuracy_metric,
    max_bootstrapped_demos=3,
    max_labeled_demos=3,
)

optimized_module = base_module

print("\n✓ Módulo otimizado pronto para uso")


Train: 8 exemplos | Val: 2 exemplos


 38%|███▊      | 3/8 [00:06<00:10,  2.10s/it]

Bootstrapped 3 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.

✓ Módulo otimizado pronto para uso


## 11. Avaliação do Módulo Otimizado

In [39]:
def evaluate_module(module: dspy.Module, examples: list, metric) -> dict:
    """Avalia um módulo em um conjunto de exemplos e retorna métricas agregadas."""
    scores = []
    details = []

    for ex in examples:
        try:
            # Chama o módulo base corretamente, respeitando sua arquitetura e assinatura
            pred = module(schema=ex.schema, question=ex.question)
        except Exception as e:
            pred = dspy.Prediction(sql_query="")

        score = metric(ex, pred)
        scores.append(score)
        details.append({
            "question": ex.question[:55] + "…",
            "score":    score,
            "sql":      getattr(pred, "sql_query", "")[:60] + "…",
        })

    return {
        "mean_score": round(sum(scores) / len(scores), 3) if scores else 0.0,
        "max_score":  round(max(scores), 3) if scores else 0.0,
        "min_score":  round(min(scores), 3) if scores else 0.0,
        "details":    details,
    }

print("=== Avaliação no conjunto de validação ===")
# Agora base_module ou optimized_module serão avaliados da maneira certa
results_eval = evaluate_module(optimized_module, val_set, sql_accuracy_metric)
print(f"  Média:  {results_eval['mean_score']}")
print(f"  Máximo: {results_eval['max_score']}")
print(f"  Mínimo: {results_eval['min_score']}")
print()
print(tabulate(
    [(d["question"], d["score"]) for d in results_eval["details"]],
    headers=["Questão", "Score"],
    tablefmt="rounded_outline"
))

=== Avaliação no conjunto de validação ===
  Média:  0.54
  Máximo: 0.96
  Mínimo: 0.12

╭───────────────────────────────────────────────────────┬─────────╮
│ Questão                                               │   Score │
├───────────────────────────────────────────────────────┼─────────┤
│ Show total project budget per department…             │    0.96 │
│ Who is the highest paid employee in each department?… │    0.12 │
╰───────────────────────────────────────────────────────┴─────────╯


## 12. Salvar e Carregar Módulo Otimizado

In [40]:
# Salva pesos otimizados (few-shot demos + instruções)
optimized_module.save("optimized_sql_generator.json")
print("✓ Módulo salvo em optimized_sql_generator.json")

# Para recarregar:
loaded = SQLGeneratorSimple()
loaded.load("optimized_sql_generator.json")


✓ Módulo salvo em optimized_sql_generator.json


## 13. Rodando o Agente Completo

In [41]:
def print_result(entry: dict) -> None:
    """Formata e imprime um resultado do agente."""
    status = "✓" if not entry.get("error") else "✗"
    cache  = " [cache]" if entry.get("from_cache") else ""
    print(f"\n{'='*60}")
    print(f"{status} [{entry.get('complexity','?').upper()}] {entry['question']}{cache}")
    print(f"   SQL      : {entry.get('sql_query','N/A')[:80]}")
    print(f"   Estratégia: {entry.get('strategy','N/A')}")
    if entry.get("error"):
        print(f"   Erro     : {entry['error']}")
    elif entry.get("results") is not None:
        print(f"   Linhas   : {entry.get('result_count', len(entry['results']))}")
    if entry.get("answer"):
        print(f"   Resposta : {entry['answer']}")
    print(f"   Tempo    : {entry.get('elapsed_ms', '?')} ms")


# Conjunto de testes cobrindo diferentes complexidades
test_questions = [
    # Simple
    "How many employees work in Engineering?",
    "What is the average salary of all employees?",
    # Medium
    "Show me all projects with a budget greater than 100000",
    "Which employees have been hired since 2022?",
    "What is the total salary cost per department?",
    # Complex
    "List all managers and the total budget of their department's projects",
    "Which department has the highest ratio of project budget to headcount?",
]

import subprocess
import time
import requests

def garantir_servidor_ollama():
    """Testa se o Ollama está rodando. Se não estiver, força a inicialização."""
    try:
        # Tenta contato com a API
        requests.get("http://localhost:11434/", timeout=2)
        print("✓ Servidor Ollama já está online.")
    except requests.exceptions.ConnectionError:
        print("⚠ Servidor Ollama offline. Reerguendo processo...")
        # Inicia o servidor
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        time.sleep(3) # Aguarda 3 segundos para o serviço respirar
        print("✓ Servidor Ollama reiniciado.")

# Garante que o motor do Ollama está ligado
garantir_servidor_ollama()

# Re-instancia o agente -> força o python a puxar as classes corrigidas da memória recente.
agent = SQLAgent(use_cache=True)
agent.cache.clear() # Limpa resquícios de falhas anteriores

# Roda as perguntas
print("\nIniciando testes...")
for q in test_questions:
    result = agent(q, interpret=True)
    print_result(result)

✓ Servidor Ollama já está online.

Iniciando testes...

✓ [MEDIUM] How many employees work in Engineering?
   SQL      : SELECT COUNT(T1.id) FROM employees AS T1 INNER JOIN departments AS T2 ON T1.depa
   Estratégia: simple
   Linhas   : 1
   Resposta : There are 4 employees working in the Engineering department.
   Tempo    : 7339 ms

✓ [MEDIUM] What is the average salary of all employees?
   SQL      : SELECT AVG(salary) FROM employees
   Estratégia: simple
   Linhas   : 1
   Resposta : The average salary of all employees is $79,500.00.
   Tempo    : 6679 ms

✓ [MEDIUM] Show me all projects with a budget greater than 100000
   SQL      : SELECT * FROM projects WHERE budget > 100000 LIMIT 100
   Estratégia: simple
   Linhas   : 3
   Resposta : There are three projects with a budget greater than $100,000: "API Development" (budget $150,000), "Mobile App" (budget $200,000), and "Data Warehouse" (budget $180,000).
   Tempo    : 6719 ms

✓ [MEDIUM] Which employees have been hired since 20

ERROR:dspy_agent:Erro no fluxo: Falha mesmo após refinamento: no such column: e.id



✗ [COMPLEX] Which department has the highest ratio of project budget to headcount?
   SQL      : SELECT d.name AS department_name, SUM(p.budget) / COUNT(e.id) AS budget_to_headc
   Estratégia: simple+refined_failed
   Erro     : Falha mesmo após refinamento: no such column: e.id
   Resposta : Não foi possível resolver a consulta: Falha mesmo após refinamento: no such column: e.id
   Tempo    : 9823 ms


## 14. Demonstração de Cache

In [29]:
print("=== Teste de Cache ===")
q = "How many employees work in Engineering?"

print("1ª chamada (sem cache):")
r1 = agent.forward(q)
print(f"  Tempo: {r1['elapsed_ms']} ms | From cache: {r1['from_cache']}")

print("\n2ª chamada (deve vir do cache):")
r2 = agent.forward(q)
print(f"  Tempo: {r2['elapsed_ms']} ms | From cache: {r2['from_cache']}")


2026/05/06 17:25:15 WARNING dspy.primitives.module: Calling module.forward(...) on SQLAgent directly is discouraged. Please use module(...) instead.
2026/05/06 17:25:15 WARNING dspy.primitives.module: Calling module.forward(...) on SQLAgent directly is discouraged. Please use module(...) instead.


=== Teste de Cache ===
1ª chamada (sem cache):
  Tempo: 11381 ms | From cache: True

2ª chamada (deve vir do cache):
  Tempo: 11381 ms | From cache: True


## 15. Histórico de Execução

In [30]:
def show_history(agent) -> None:
    """Exibe o histórico de queries em formato tabular com proteção de chaves."""
    rows = []
    for i, entry in enumerate(agent.history, 1):
        # Usa .get() com valor padrão para evitar KeyError se a memória tiver "lixo"
        question = entry.get("question", "Questão não registrada")
        rows.append([
            i,
            question[:45] + "…",
            entry.get("complexity", "?"),
            entry.get("strategy", "?"),
            "✓" if not entry.get("error") else "✗",
            f"{entry.get('elapsed_ms','?')} ms",
        ])

    print(tabulate(
        rows,
        headers=["#", "Questão", "Complexidade", "Estratégia", "Status", "Tempo"],
        tablefmt="rounded_outline"
    ))

show_history(agent)


╭─────┬────────────────────────────────────────────────┬────────────────┬───────────────────────┬──────────┬──────────╮
│   # │ Questão                                        │ Complexidade   │ Estratégia            │ Status   │ Tempo    │
├─────┼────────────────────────────────────────────────┼────────────────┼───────────────────────┼──────────┼──────────┤
│   1 │ Questão não registrada…                        │ ?              │ ?                     │ ✓        │ ? ms     │
│   2 │ Questão não registrada…                        │ ?              │ ?                     │ ✓        │ ? ms     │
│   3 │ Questão não registrada…                        │ ?              │ ?                     │ ✓        │ ? ms     │
│   4 │ Questão não registrada…                        │ ?              │ ?                     │ ✓        │ ? ms     │
│   5 │ How many employees work in Engineering?…       │ medium         │ simple                │ ✓        │ 11381 ms │
│   6 │ Questão não registrada…         

## 16. Interface Interativa (REPL)

In [31]:
def interactive_agent(agent: SQLAgent) -> None:
    """
    Loop interativo para consultas em linguagem natural.
    Digite 'sair' para encerrar, 'historico' para ver o log, 'cache clear' para limpar.
    """
    print("\n🤖 SQL Agent interativo — digite sua pergunta em inglês")
    print("   Comandos: 'sair' | 'historico' | 'cache clear'\n")

    while True:
        try:
            question = input("Você: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nEncerrando agente.")
            break

        if not question:
            continue
        if question.lower() in ("sair", "exit", "quit"):
            print("Até logo!")
            break
        if question.lower() == "historico":
            show_history(agent)
            continue
        if question.lower() == "cache clear":
            agent.cache.clear()
            print("Cache limpo.")
            continue

        result = agent.forward(question, interpret=True)
        print_result(result)

# Uso interativo:
interactive_agent(agent)
print("ℹ Descomente a última linha para ativar o modo interativo.")



🤖 SQL Agent interativo — digite sua pergunta em inglês
   Comandos: 'sair' | 'historico' | 'cache clear'

Você: historico
╭─────┬────────────────────────────────────────────────┬────────────────┬───────────────────────┬──────────┬──────────╮
│   # │ Questão                                        │ Complexidade   │ Estratégia            │ Status   │ Tempo    │
├─────┼────────────────────────────────────────────────┼────────────────┼───────────────────────┼──────────┼──────────┤
│   1 │ Questão não registrada…                        │ ?              │ ?                     │ ✓        │ ? ms     │
│   2 │ Questão não registrada…                        │ ?              │ ?                     │ ✓        │ ? ms     │
│   3 │ Questão não registrada…                        │ ?              │ ?                     │ ✓        │ ? ms     │
│   4 │ Questão não registrada…                        │ ?              │ ?                     │ ✓        │ ? ms     │
│   5 │ How many employees work in En

2026/05/06 17:25:44 WARNING dspy.primitives.module: Calling module.forward(...) on SQLAgent directly is discouraged. Please use module(...) instead.



✓ [MEDIUM] How many employees work in Engineering? [cache]
   SQL      : SELECT COUNT(e.id) FROM employees e INNER JOIN departments d ON e.department = d
   Estratégia: simple
   Linhas   : 1
   Resposta : There are 4 employees working in the Engineering department.
   Tempo    : 11381 ms
Você: How many employees work in Sales?


2026/05/06 17:26:13 WARNING dspy.primitives.module: Calling module.forward(...) on SQLAgent directly is discouraged. Please use module(...) instead.



✓ [MEDIUM] How many employees work in Sales?
   SQL      : SELECT COUNT(T1.id) FROM employees AS T1 INNER JOIN departments AS T2 ON T1.depa
   Estratégia: simple
   Linhas   : 1
   Resposta : There are 2 employees working in the Sales department.
   Tempo    : 7681 ms

Encerrando agente.
ℹ Descomente a última linha para ativar o modo interativo.


## 17. Resumo das Melhorias Implementadas

| Área | O que foi feito |
|---|---|
| **Signatures** | 5 signatures tipadas: `GenerateSQL`, `RefineSQL`, `InterpretResults`, `ValidateSQL`, `ClassifyQuestion` |
| **Modules** | `SQLValidator`, `SQLGeneratorSimple` (CoT), `SQLGeneratorMultiChain` (MultiChainComparison, M=3), `SQLInterpreter` |
| **MultiChain** | Sim, funciona — gera 3 candidatos e escolhe o melhor; ideal para queries medium/complex |
| **Otimizadores** | `BootstrapFewShot`, `BootstrapFewShotWithRandomSearch`, `MIPROv2` (quando disponível) |
| **Banco de dados** | Expandido para 10 funcionários, 4 departamentos, 6 projetos |
| **Cache** | MD5 hash da pergunta → evita reprocessamento |
| **Histórico** | Log completo com timestamp, complexidade, estratégia, tempo e status |
| **Classificação automática** | Cada query é classificada em *simple/medium/complex* e roteada para o módulo adequado |
| **Avaliação** | Métrica composta (execução + colunas + keywords) + função `evaluate_module` |
| **Persistência** | `module.save()` / `module.load()` para reutilizar o módulo otimizado |
| **REPL interativo** | Loop `interactive_agent()` com comandos de histórico e limpeza de cache |

### Sobre MultiChainComparison
`dspy.MultiChainComparison` funciona gerando **M chains of thought independentes** para o mesmo signature e então aplicando um modelo de comparação para escolher a resposta mais consistente. Isso é análogo a *self-consistency* em chain-of-thought prompting — reduz variância e aumenta confiabilidade especialmente em queries SQL complexas com JOINs e subqueries.
